In [1]:
!date

Wed Sep 16 08:19:21 PDT 2026


In [2]:

import subprocess
from concurrent.futures import ProcessPoolExecutor
import os

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from tqdm import tqdm

projdir = '/u/project/cluo/terencew/claude/project_ideas/asm_lr_hprc2'


In [3]:

### validation-scale sample set -- 2 donors per superpop, 5 superpops = 10 donors, 20 (sample,hap) files.
### chosen because this compute session only has nproc=1 -- a full-scan-per-file approach (see
### below, no subsampling needed) runs ~90s/file, so 20 files serially is ~30min, tractable here.
### scaling to n=230 needs a qsub array (see notes at the end), not more donors run serially in-notebook.
samples = ['HG00097','HG00099','HG00408','HG00423','HG00639','HG00642','HG01884','HG01891','HG02602','HG02698']

manifest = pd.read_csv(f'{projdir}/tsv/meta/hprc2_sample_manifest.tsv', sep='\t')
manifest = manifest[manifest['sample_id'].isin(samples)]
manifest.head()


,sample_id,population,superpopulation,ont_hap1_modbed_mb,ont_hap2_modbed_mb,has_pacbio_methylc,has_fiberseq,has_hic,has_isoseq_expression,has_harmonized_sup5_basecall,had_guppy_raw,had_dorado06_raw,assembly_naming,assembly_hap1_url,assembly_hap2_url,in_asm_lr_18donor_cohort,in_asm_lr_30donor_plan
0,HG00097,GBR,EUR,2343.2,2312.9,True,False,True,True,True,False,True,hap,https://human-pangenomics.s3.amazonaws.com/wor...,https://human-pangenomics.s3.amazonaws.com/wor...,False,False
1,HG00099,GBR,EUR,2632.4,2620.3,True,False,True,True,True,True,False,hap,https://human-pangenomics.s3.amazonaws.com/wor...,https://human-pangenomics.s3.amazonaws.com/wor...,False,False
19,HG00408,CHS,EAS,3210.9,3181.8,True,False,True,True,True,True,False,trio,https://human-pangenomics.s3.amazonaws.com/wor...,https://human-pangenomics.s3.amazonaws.com/wor...,False,False
20,HG00423,CHS,EAS,2726.9,2778.9,True,False,True,True,True,True,False,trio,https://human-pangenomics.s3.amazonaws.com/wor...,https://human-pangenomics.s3.amazonaws.com/wor...,False,False
28,HG00639,PUR,AMR,2692.8,2753.3,True,False,True,True,True,True,False,trio,https://human-pangenomics.s3.amazonaws.com/wor...,https://human-pangenomics.s3.amazonaws.com/wor...,False,False


In [4]:
manifest.shape

(10, 17)

In [5]:

### sex covariate: NOT in the HPRC2 manifest at all -- the only public source found is the
### 1000G/IGSR pedigree file (cached locally, results/qc/data/g1k_3202_samples_ped_population.txt).
### age is NOT a column in that file either, and no age field was found anywhere in HPRC2's
### public metadata or manifest -- age is not usable as a covariate for this cohort from public
### data. no passage-number / LCL-culture metadata field was found anywhere either.
ped = pd.read_csv(f'{projdir}/results/qc/data/g1k_3202_samples_ped_population.txt', sep=' ')
ped = ped[ped['SampleID'].isin(samples)][['SampleID','Sex']]
ped.head()


,SampleID,Sex
1,HG00097,2
2,HG00099,2
190,HG00408,2
198,HG00423,2
314,HG00639,2


In [6]:

covariates = manifest.merge(ped, left_on='sample_id', right_on='SampleID')
covariates['sex'] = covariates['Sex'].map({1: 'male', 2: 'female'})
covariates = covariates[['sample_id','population','superpopulation','sex',
                          'has_harmonized_sup5_basecall','had_guppy_raw','had_dorado06_raw']]
covariates.head(10)


,sample_id,population,superpopulation,sex,has_harmonized_sup5_basecall,had_guppy_raw,had_dorado06_raw
0,HG00097,GBR,EUR,female,True,False,True
1,HG00099,GBR,EUR,female,True,True,False
2,HG00408,CHS,EAS,female,True,True,False
3,HG00423,CHS,EAS,female,True,True,False
4,HG00639,PUR,AMR,female,True,True,False
5,HG00642,PUR,AMR,male,True,True,False
6,HG01884,ACB,AFR,female,True,True,False
7,HG01891,ACB,AFR,female,True,True,False
8,HG02602,PJL,SAS,male,True,True,False
9,HG02698,PJL,SAS,male,True,True,False


In [7]:

### global per-haplotype methylation -- full scan of each modbed, not a subsample. modbed col7 =
### comma-separated methylated-CpG-call offsets, col8 = unmethylated (docs/data_sources.md SS5),
### so total-calls-in-col7 / total-calls-in-(col7+col8) is the global fraction methylated for
### that (sample, hap), no per-position aggregation or liftover needed -- decompression (not
### parsing) is the bottleneck (~45s/file just for zcat), so a full scan is barely slower than a
### subsample would have been and is exact rather than estimated.
AWK_SCRIPT = '{nm=($7==\"\"||$7==\".\"||$7==\"-\")?0:gsub(\",\",\",\",$7)+1; nu=($8==\"\"||$8==\".\"||$8==\"-\")?0:gsub(\",\",\",\",$8)+1; tm+=nm; tu+=nu; nr++} END{print tm\"\\t\"tu\"\\t\"nr}'

def global_meth_counts(sample_hap):
    sample, hap = sample_hap
    path = f'{projdir}/data/modbed/{sample}_hap{hap}.modbed.gz'
    p1 = subprocess.Popen(['zcat', path], stdout=subprocess.PIPE)
    p2 = subprocess.run(['awk', '-F\t', AWK_SCRIPT], stdin=p1.stdout, capture_output=True, text=True)
    p1.stdout.close()
    tm, tu, nr = p2.stdout.strip().split('\t')
    return sample, hap, int(tm), int(tu), int(nr)


In [8]:

tasks = [(s, h) for s in samples for h in (1, 2)]
results = []
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    for r in tqdm(ex.map(global_meth_counts, tasks), total=len(tasks)):
        results.append(r)


100%|██████████| 20/20 [04:44<00:00, 14.23s/it]


In [9]:

meth = pd.DataFrame(results, columns=['sample_id','hap','n_meth','n_unmeth','n_reads'])
meth['frac_meth'] = meth['n_meth'] / (meth['n_meth'] + meth['n_unmeth'])
meth.head(20)


,sample_id,hap,n_meth,n_unmeth,n_reads,frac_meth
0,HG00097,1,531491673,310596693,1532307,0.631159
1,HG00097,2,521479385,311345741,1544277,0.626157
2,HG00099,1,636242710,311852572,2129416,0.671075
3,HG00099,2,629642090,316319742,2151600,0.665610
4,HG00408,1,744902636,410877694,2249331,0.644502
5,HG00408,2,736096456,410730307,2243454,0.641855
6,HG00423,1,660620855,317516923,1766873,0.675386
7,HG00423,2,675416228,325228477,1785000,0.674981
8,HG00639,1,641886700,324524901,1758374,0.664196
9,HG00639,2,655655972,335431047,1782038,0.661552


In [10]:
meth.shape

(20, 6)

In [11]:

df = meth.merge(covariates, on='sample_id')
df['hap'] = df['hap'].astype(str)
df.head(20)


,sample_id,hap,n_meth,n_unmeth,n_reads,frac_meth,population,superpopulation,sex,has_harmonized_sup5_basecall,had_guppy_raw,had_dorado06_raw
0,HG00097,1,531491673,310596693,1532307,0.631159,GBR,EUR,female,True,False,True
1,HG00097,2,521479385,311345741,1544277,0.626157,GBR,EUR,female,True,False,True
2,HG00099,1,636242710,311852572,2129416,0.671075,GBR,EUR,female,True,True,False
3,HG00099,2,629642090,316319742,2151600,0.665610,GBR,EUR,female,True,True,False
4,HG00408,1,744902636,410877694,2249331,0.644502,CHS,EAS,female,True,True,False
5,HG00408,2,736096456,410730307,2243454,0.641855,CHS,EAS,female,True,True,False
6,HG00423,1,660620855,317516923,1766873,0.675386,CHS,EAS,female,True,True,False
7,HG00423,2,675416228,325228477,1785000,0.674981,CHS,EAS,female,True,True,False
8,HG00639,1,641886700,324524901,1758374,0.664196,PUR,AMR,female,True,True,False
9,HG00639,2,655655972,335431047,1782038,0.661552,PUR,AMR,female,True,True,False


In [12]:

outdir = f'{projdir}/results/qc/data'
df.to_csv(f'{outdir}/global_methylation_covariates_n10_validation.tsv', sep='\t', index=False)
df[['sample_id','hap','frac_meth','n_reads']]


,sample_id,hap,frac_meth,n_reads
0,HG00097,1,0.631159,1532307
1,HG00097,2,0.626157,1544277
2,HG00099,1,0.671075,2129416
3,HG00099,2,0.665610,2151600
4,HG00408,1,0.644502,2249331
5,HG00408,2,0.641855,2243454
6,HG00423,1,0.675386,1766873
7,HG00423,2,0.674981,1785000
8,HG00639,1,0.664196,1758374
9,HG00639,2,0.661552,1782038


In [13]:

### variance-explained model -- n=10 donors (20 hap-level observations) is a validation-scale
### pilot, NOT a properly powered fit (5 superpop levels + sex + 2 usable binary basecall-history
### flags is close to n params ~ n obs at this scale -- R^2 here will be inflated/overfit, reported
### only to confirm the pipeline runs end to end, not as a real estimate of variance explained).
### had_dorado06_raw dropped -- collinear with has_harmonized_sup5_basecall in this small sample.
model = smf.ols('frac_meth ~ C(superpopulation) + C(sex) + C(hap) + C(has_harmonized_sup5_basecall)',
                 data=df).fit()
model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              frac_meth   R-squared:                       0.525
Model:                            OLS   Adj. R-squared:                  0.306
Method:                 Least Squares   F-statistic:                     2.398
Date:                Wed, 16 Sep 2026   Prob (F-statistic):             0.0878
Time:                        08:24:13   Log-Likelihood:                 49.473
No. Observations:                  20   AIC:                            -84.95
Df Residuals:                      13   BIC:                            -77.98
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
===========================================================================================================
                                              coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept                                   0.3417      0.007     49.333      0.000       0.327       0.357
C(superpopulation)[T.AMR]                  -0.0195      0.022     -0.889      0.390      -0.067       0.028
C(superpopulation)[T.EAS]                  -0.0232      0.018     -1.296      0.218      -0.062       0.015
C(superpopulation)[T.EUR]                  -0.0339      0.018     -1.893      0.081      -0.072       0.005
C(superpopulation)[T.SAS]                  -0.0874      0.031     -2.820      0.014      -0.154      -0.020
C(sex)[T.male]                              0.0333      0.025      1.317      0.211      -0.021       0.088
C(hap)[T.2]                                -0.0022      0.011     -0.195      0.848      -0.027       0.022
C(has_harmonized_sup5_basecall)[T.True]     0.3417      0.007     49.333      0.000       0.327       0.357
==============================================================================
Omnibus:                        0.008   Durbin-Watson:                   1.507
Prob(Omnibus):                  0.996   Jarque-Bera (JB):                0.204
Skew:                           0.005   Prob(JB):                        0.903
Kurtosis:                       2.505   Cond. No.                     1.05e+17
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 4.59e-33. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [14]:

sm.stats.anova_lm(model, typ=2)


,sum_sq,df,F,PR(>F)
C(superpopulation),0.008539,4.0,3.336753,4.351301e-02
C(sex),0.001109,1.0,1.733750,2.106737e-01
C(hap),0.000024,1.0,0.038206,8.480530e-01
C(has_harmonized_sup5_basecall),1.557060,1.0,2433.768863,3.567834e-16
Residual,0.008317,13.0,NaN,NaN


In [15]:
### FULL-COHORT RUN -- scaled from the n=10 validation above via a real qsub array
### (scripts/qsub/M01_global_methylation_array.sh, job 14765944), one task per donor, full
### genome-wide zcat+awk scan per haplotype (not subsampled -- the validation already showed a
### full scan is barely slower than subsampling would be, decompression dominates). Output:
### results/qc/data/per_sample_global_methylation/<sample>.tsv, one skip-if-exists file per donor.
import glob

per_sample_files = glob.glob(f'{projdir}/results/qc/data/per_sample_global_methylation/*.tsv')
len(per_sample_files)


219

In [16]:
meth_full = pd.concat([pd.read_csv(f, sep='\t') for f in per_sample_files], ignore_index=True)
meth_full = meth_full.rename(columns={'sample': 'sample_id'})
meth_full['frac_meth'] = meth_full['n_meth'] / (meth_full['n_meth'] + meth_full['n_unmeth'])
meth_full.head()


,sample_id,hap,n_meth,n_unmeth,n_reads,frac_meth
0,HG00097,1,531491673,310596693,1532307,0.631159
1,HG00097,2,521479385,311345741,1544277,0.626157
2,HG00099,1,636242710,311852572,2129416,0.671075
3,HG00099,2,629642090,316319742,2151600,0.665610
4,HG00126,1,727083303,373674535,1491329,0.660530


In [17]:
meth_full.shape

(438, 6)

In [18]:
### passage number -- coordinator flagged this mid-task: IS publicly available, missed by the
### earlier "no passage metadata found anywhere" search. reference/hprc2/hprc2_supp.xlsx, sheet
### S15 "Cell Line Karyotypes", column "Cell Line Passage*" (post-establishment / post-receipt-
### at-Coriell passage count). 234/235 donors, mostly 5, range 3-12, 28 NaN (not karyotyped --
### sheet's own header note says HPRC-Plus/HPP samples didn't generally get karyotyped as part
### of QC, so NaN here plausibly means "not measured", not "zero").
s15 = pd.read_excel(f'{projdir}/reference/hprc2/hprc2_supp.xlsx', sheet_name='S15', header=1)
s15 = s15[['Cell Line Sample ID', 'Cell Line Passage*']].rename(
    columns={'Cell Line Sample ID': 'sample_id', 'Cell Line Passage*': 'passage'})
s15['passage'].value_counts(dropna=False).sort_index()


3.0       6
4.0       1
5.0     170
6.0       6
7.0       3
8.0      16
9.0       1
11.0      2
12.0      1
NaN      28
Name: passage, dtype: int64

In [19]:
manifest_full = pd.read_csv(f'{projdir}/tsv/meta/hprc2_sample_manifest.tsv', sep='\t')
ped_full = pd.read_csv(f'{projdir}/results/qc/data/g1k_3202_samples_ped_population.txt', sep=' ')
ped_full = ped_full[['SampleID','Sex']]

df_full = meth_full.merge(manifest_full[['sample_id','population','superpopulation']], on='sample_id', how='left')
df_full = df_full.merge(ped_full, left_on='sample_id', right_on='SampleID', how='left')
df_full['sex'] = df_full['Sex'].map({1: 'male', 2: 'female'})
df_full = df_full.merge(s15, on='sample_id', how='left')
df_full['hap'] = df_full['hap'].astype(str)
df_full.head()


,sample_id,hap,n_meth,n_unmeth,n_reads,frac_meth,population,superpopulation,SampleID,Sex,sex,passage
0,HG00097,1,531491673,310596693,1532307,0.631159,GBR,EUR,HG00097,2.0,female,5.0
1,HG00097,2,521479385,311345741,1544277,0.626157,GBR,EUR,HG00097,2.0,female,5.0
2,HG00099,1,636242710,311852572,2129416,0.671075,GBR,EUR,HG00099,2.0,female,5.0
3,HG00099,2,629642090,316319742,2151600,0.665610,GBR,EUR,HG00099,2.0,female,5.0
4,HG00126,1,727083303,373674535,1491329,0.660530,GBR,EUR,HG00126,1.0,male,5.0


In [20]:
df_full.shape

(438, 12)

In [21]:
### real, full-cohort methylation range (vs. the n=10 validation's 0.620-0.720)
df_full['frac_meth'].describe()


count    438.000000
mean       0.646941
std        0.034658
min        0.520835
25%        0.625700
50%        0.648456
75%        0.671306
max        0.725917
Name: frac_meth, dtype: float64

In [22]:
outdir = f'{projdir}/results/qc/data'
df_full.to_csv(f'{outdir}/global_methylation_covariates_full_cohort.tsv', sep='\t', index=False)
df_full[['sample_id','hap','frac_meth','superpopulation','sex','passage']].to_csv(
    f'{outdir}/global_methylation_covariates_full_cohort.tsv', sep='\t', index=False)


In [23]:
### baseline model (ancestry + sex + hap) -- no passage -- at full n this time, not the n=10
### validation's near-singular fit.
model_base = smf.ols('frac_meth ~ C(superpopulation) + C(sex) + C(hap)', data=df_full, missing='drop').fit()
model_base.rsquared


0.12463573276505469

In [24]:
### + passage. Restricting to donors with a non-null passage value (the karyotyping-coverage
### gap noted above) -- same row set used for both models below so the R^2 comparison is apples-
### to-apples, not confounded by the baseline model silently having more rows.
df_pass = df_full.dropna(subset=['passage']).copy()
model_base_matched = smf.ols('frac_meth ~ C(superpopulation) + C(sex) + C(hap)', data=df_pass).fit()
model_plus_passage = smf.ols('frac_meth ~ C(superpopulation) + C(sex) + C(hap) + passage', data=df_pass).fit()
print('n rows (passage non-null):', len(df_pass))
print('baseline R2:', model_base_matched.rsquared)
print('+passage R2:', model_plus_passage.rsquared)


n rows (passage non-null): 314
baseline R2: 0.1605902404127798
+passage R2: 0.16829751230921342


In [25]:
### F-test: does adding passage explain significantly more variance than ancestry+sex+hap alone
anova_compare = sm.stats.anova_lm(model_base_matched, model_plus_passage)
anova_compare


,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,297.0,0.261271,0.0,NaN,NaN,NaN
1,296.0,0.258872,1.0,0.002399,2.742991,0.098741


In [26]:
### passage range actually testable in THIS cohort's modbed-downloaded subset -- the coordinator's
### note flagged that passage>=8 donors currently lack modbed data (the same ~27-sample
### unprocessed stratum noted elsewhere in this project's docs), so check what range df_pass
### actually covers before over-interpreting the F-test above.
df_pass['passage'].value_counts().sort_index()


3.0     12
4.0      2
5.0    290
6.0      8
7.0      2
Name: passage, dtype: int64

In [27]:
### passage x superpopulation crosstab -- checking whether passage is confounded with ancestry
### at full scale the way the coordinator's small (n=15) rerun found it was (that rerun's
### passage-carrying addition set happened to be all-EUR).
pd.crosstab(df_pass['superpopulation'], df_pass['passage'])


passage,3.0,5.0,6.0,7.0
superpopulation,,,,
AFR,0,72,2,0
AMR,0,72,2,0
EAS,0,56,0,0
EUR,12,26,4,2
SAS,0,56,0,0


In [28]:
model_plus_passage.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              frac_meth   R-squared:                       0.168
Model:                            OLS   Adj. R-squared:                  0.149
Method:                 Least Squares   F-statistic:                     8.557
Date:                Wed, 16 Sep 2026   Prob (F-statistic):           1.49e-09
Time:                        08:24:24   Log-Likelihood:                 643.05
No. Observations:                 304   AIC:                            -1270.
Df Residuals:                     296   BIC:                            -1240.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     0.6194      0.020     31.121      0.000       0.580       0.659
C(superpopulation)[T.AMR]     0.0102      0.005      2.082      0.038       0.001       0.020
C(superpopulation)[T.EAS]     0.0050      0.005      0.940      0.348      -0.005       0.015
C(superpopulation)[T.EUR]    -0.0218      0.006     -3.744      0.000      -0.033      -0.010
C(superpopulation)[T.SAS]     0.0111      0.005      2.090      0.037       0.001       0.022
C(sex)[T.male]                0.0074      0.003      2.131      0.034       0.001       0.014
C(hap)[T.2]                  -0.0028      0.003     -0.822      0.412      -0.009       0.004
passage                       0.0064      0.004      1.656      0.099      -0.001       0.014
==============================================================================
Omnibus:                       16.597   Durbin-Watson:                   0.987
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               18.055
Skew:                          -0.531   Prob(JB):                     0.000120
Kurtosis:                       3.546   Cond. No.                         61.9
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [29]:
### full-cohort verdict, in place of the n=10 validation's "not properly powered yet" caveat
### above: with real cluster-scale n and the passage covariate the coordinator surfaced mid-task,
### state plainly whether the epigenome lead's "not fully explained by common covariates"
### observation replicates, and whether passage closes the gap or (like the coordinator's n=15
### rerun found) is itself confounded with ancestry in the currently-downloaded subset.
print(f"baseline (ancestry+sex+hap) R2, matched rows: {model_base_matched.rsquared:.3f}")
print(f"+passage R2: {model_plus_passage.rsquared:.3f}")
print(f"passage F-test p-value: {anova_compare['Pr(>F)'][1]:.4g}")


baseline (ancestry+sex+hap) R2, matched rows: 0.161
+passage R2: 0.168
passage F-test p-value: 0.09874


In [30]:
!date

Wed Sep 16 08:24:24 PDT 2026
